In [0]:
from pyspark.sql.functions import *
from delta.tables import DeltaTable
from pyspark.sql.window import Window 

In [0]:
%run /Workspace/consolidated_pipeline/02_dimension_data_processing/utilities

In [0]:
dbutils.widgets.text('catalog','fmcg','Catalog')
dbutils.widgets.text('data_source','orders','Data Source')

catalog=dbutils.widgets.get('catalog')
data_source=dbutils.widgets.get('data_source')

In [0]:
base_path=f"/Volumes/{catalog}/{bronze_schema}/sports_volume/{data_source}"
landing_path=f'{base_path}/landing'
processed_path=f'{base_path}/processed'


In [0]:
# define the tables
bronze_table=f"{catalog}.{bronze_schema}.{data_source}"
silver_table=f"{catalog}.{silver_schema}.{data_source}"
gold_table=f"{catalog}.{gold_schema}.sb_fact_{data_source}"

In [0]:
df=spark.read.options(header=True,inferSchema=True).csv(f'{landing_path}/*.csv').withColumn('read_time_stamp',
                                                                                           current_timestamp())
display(df)

In [0]:
df.count()

In [0]:
# df.write.format('delta').mode('append').option('delta.enableChangeDataFeed','true').saveAsTable(bronze_table)

In [0]:
files=dbutils.fs.ls(landing_path)
for file in files:
    dbutils.fs.mv(file.path,
                  f"{processed_path}/{file.name}",True)
                    

In [0]:
df_silver=spark.sql(f'select * from {bronze_table}')
display(df_silver)

In [0]:
df_silver=df_silver.filter(col('order_qty').isNotNull())
df_silver=df_silver.withColumn('customer_id',
                               when(
                                   col('customer_id').rlike("^[0-9]+$"),col('customer_id')
                               ).otherwise(999999).cast('string')
                               )
df_silver=df_silver.withColumn('order_placement_date',regexp_replace('order_placement_date',r'^[A-Za-z]+,\s*',""))

df_silver=df_silver.withColumn('order_placement_date', coalesce(
try_to_date('order_placement_date','yyyy/MM/dd'),
try_to_date('order_placement_date','dd-MM-yyyy'),
 try_to_date('order_placement_date','dd/MM/yyyy'),
  try_to_date('order_placement_date','MMMM dd, yyyy')                                                                ))

df_silver=df_silver.dropDuplicates(['order_id','order_placement_date','customer_id','product_id','order_qty'])
df_silver=df_silver.withColumn('product_id', col('product_id').cast('string'))

In [0]:
display(df_silver)

In [0]:
df_silver.agg(min("order_placement_date").alias('min_date'),max("order_placement_date").alias('max_date')).show()

In [0]:
df_products=spark.table('fmcg.silver.products')

In [0]:
display(df_products.limit(5))

In [0]:
df_joined=df_silver.join(df_products,on="product_id",how='inner').select(df_silver["*"],df_products['product_code'])
display(df_joined.limit(10))

In [0]:
df_silver.count()


In [0]:
df_joined.count()
 

In [0]:
if not  (spark.catalog.tableExists(silver_table)):
    df_joined.write.format('delta').option('delta.enableChangeDataFeed','true').option('mergeSchema','true').mode('overwrite').saveAsTable(silver_table)
else:
    silver_delta=DeltaTable.forName(spark,silver_table)
    silver_delta.alias('silver').merge(
        source=df_joined.alias('source'),
        condition='''
        silver.order_placement_date=source.order_placement_date
         and silver.order_id=source.order_id 
         and silver.product_code=source.product_code 
         and silver.customer_id=source.customer_id
        '''
    ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

Gold

In [0]:
df_gold=spark.sql(f"select order_id, order_placement_date as date, customer_id as customer_code,product_code, product_id,order_qty as sold_quantity from {silver_table}")
display(df_gold.limit(20))

In [0]:
df_gold.count()

In [0]:
if not (spark.catalog.tableExists(gold_table)):
    df_gold.write.format('delta').option('mergeSchema','true').option('delta.enableChangeDataFeed','true').mode('overwrite').saveAsTable(gold_table)
else:
    delta_table=DeltaTable.forName(spark,gold_table)
    delta_table.alias('target').merge(
        source=df_gold.alias('source'),
        condition='''source.date=target.date
        and source.order_id=target.order_id
        and source.product_code=target.product_code
        '''
    ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

merge with parent

In [0]:
df_child=spark.sql(f"select date, product_code,customer_code,sold_quantity from {gold_table}") 
display(df_child.limit(40))

In [0]:
# changing the date to the first day of the month
df_monthly=df_child.withColumn('month_start',trunc('date','MM')).groupBy('month_start','product_code','customer_code').agg(sum('sold_quantity').alias('sold_quantity')).withColumnRenamed('month_start','date')
display(df_monthly.limit(20))

In [0]:
df_monthly.count()

In [0]:
gold_parent=DeltaTable.forName(spark,f"{catalog}.{gold_schema}.fact_orders")
gold_parent.alias('target').merge(
    source=df_monthly.alias('source'),
    condition='''
    source.date=target.date
    and source.product_code=target.product_code
    and source.customer_code=target.customer_code
    '''
    ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()